<a href="https://colab.research.google.com/github/EshikaAbbaraju/Role_Specified_Collective_Foraging_Task_Model/blob/main/Role_Specified_Collective_Foraging_Task_Model_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!git clone https://github.com/JerryGuo2001/Role_Speficifed_Collective_Foraging_Task.git
#%cd Role_Speficifed_Collective_Foraging_Task

#tune the danger from aliens
#print out snapshot labels at the top

%cd /content/Role_Speficifed_Collective_Foraging_Task

!git add .
!git commit -m "update experiment"
!git push origin main

CSV_DIR = "gridworld/"

/content/Role_Speficifed_Collective_Foraging_Task
[master (root-commit) 191fcd8] update experiment
 25 files changed, 6461 insertions(+)
 create mode 100644 .DS_Store
 create mode 100644 TexturePack/.DS_Store
 create mode 100644 TexturePack/allien.png
 create mode 100644 TexturePack/gold_mine.png
 create mode 100644 Util_ipynb/.DS_Store
 create mode 100644 Util_ipynb/random_grid_map_generator.ipynb
 create mode 100644 forager_01.gif
 create mode 100644 forager_single_01.csv
 create mode 100644 gridworld/.DS_Store
 create mode 100644 gridworld/middle_reward_middle_risk_01.csv
 create mode 100644 gridworld/middle_reward_middle_risk_02.csv
 create mode 100644 gridworld/middle_reward_middle_risk_03.csv
 create mode 100644 gridworld/middle_reward_middle_risk_04.csv
 create mode 100644 gridworld/middle_reward_middle_risk_05.csv
 create mode 100644 gridworld/middle_reward_middle_risk_06.csv
 create mode 100644 gridworld/middle_reward_middle_risk_07.csv
 create mode 100644 gridworld/middle_rew

In [ ]:
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# CSV path: try gridworld/ (repo), then current dir (e.g. if you uploaded the CSV)
CSV_DIR = "gridworld/" if os.path.isdir("gridworld") else "."

def load_env_from_csv(csv_path: str, prob_A: float = 0.8, prob_B: float = 0.5, prob_C: float = 0.2) -> pd.DataFrame:
    raw = pd.read_csv(csv_path)
    raw = raw.rename(columns={"x": "Row", "y": "Col", "mine_type": "mine_type", "alien_id": "alien_id"})
    rows = []
    for _, row in raw.iterrows():
        r, c = int(row["Row"]), int(row["Col"])
        mine_type = str(row["mine_type"]).strip() if not pd.isna(row["mine_type"]) else ""
        alien_val = row["alien_id"]
        if mine_type == "Gold Mine A": reward_prob, richness = prob_A, "high"
        elif mine_type == "Gold Mine B": reward_prob, richness = prob_B, "medium"
        elif mine_type == "Gold Mine C": reward_prob, richness = prob_C, "low"
        else: mine_type, reward_prob, richness = "", 0.0, "none"
        alien_level = 0 if (pd.isna(alien_val) or str(alien_val).strip() == "") else int(float(alien_val))
        rows.append({"Row": r, "Col": c, "Location": f"{r}:{c}", "mine_type": mine_type, "richness": richness, "reward_prob": float(reward_prob), "alien_level": alien_level})
    env = pd.DataFrame(rows)
    env.set_index(["Row", "Col"], inplace=True, drop=False)
    return env

def load_grid_from_csv(csv_path: str, prob_A: float = 0.8, prob_B: float = 0.5, prob_C: float = 0.2) -> pd.DataFrame:
    return load_env_from_csv(csv_path, prob_A=prob_A, prob_B=prob_B, prob_C=prob_C)

@dataclass
class ForagerConfig:
    total_moves: int = 5
    lambda_sec: float = 1.0   # >0 leader (explore away from security)
    w_scale: float = 1.0
    move_cost: float = 0.0
    security_pos: Optional[Tuple[int, int]] = None
    seed: Optional[int] = None

class ForagerAgent:
    def __init__(self, env: pd.DataFrame, cfg: ForagerConfig):
        self.env, self.cfg = env, cfg
        self.rng = np.random.default_rng(cfg.seed)
        max_r = int(self.env.index.get_level_values(0).max())
        max_c = int(self.env.index.get_level_values(1).max())
        self.pos = (max_r // 2, max_c // 2)
        self.security_pos = cfg.security_pos or (max_r // 2, max_c // 2)
        self.t, self.total_reward = 0, 0.0
        self.visited = {self.pos}
        self.stunned_steps = 0
        self.log, self.frames_pos, self.frames_action, self.frames_decision = [], [], [], []
        self.frames_Vdig, self.frames_Vmove, self.frames_w, self.frames_total_reward = [], [], [], []

    def current_reward_prob(self) -> float:
        return float(self.env.loc[self.pos, "reward_prob"])

    def neighbors(self) -> List[Tuple[int, int]]:
        r, c = self.pos
        return [p for p in [(r-1,c),(r+1,c),(r,c-1),(r,c+1)] if p in self.env.index]

    def Vdig(self) -> float:
        return self.current_reward_prob()

    def Vmove(self) -> float:
        return self.total_reward / float(self.t) if self.t > 0 else 0.0

    def w_t(self, Vdig_t: float, Vmove_t: float) -> float:
        delta = Vdig_t - Vmove_t
        return float(1.0 / (1.0 + np.exp(-max(self.cfg.w_scale, 1e-6) * delta)))

    def E_exploit(self, s_prime: Tuple[int, int]) -> float:
        known = [p for p in self.visited if (p in self.env.index) and (self.env.loc[p, "mine_type"] != "")]
        if not known: return 0.0
        d_min = min(abs(s_prime[0]-p[0])+abs(s_prime[1]-p[1]) for p in known)
        return max(float(self.env.loc[p, "reward_prob"]) for p in known) / (1.0 + d_min)

    def E_explore(self, s_prime: Tuple[int, int]) -> float:
        unv = [p for p in self.env.index if p not in self.visited]
        if not unv: return 0.0
        d_min = min(abs(s_prime[0]-p[0])+abs(s_prime[1]-p[1]) for p in unv)
        return 1.0 / (1.0 + d_min)

    def A_goal(self, s_prime: Tuple[int, int], w: float) -> float:
        return (1.0 - w) * self.E_exploit(s_prime) + w * self.E_explore(s_prime)

    def _log(self, **kw):
        kw.setdefault("step", self.t); kw.setdefault("row", self.pos[0]); kw.setdefault("col", self.pos[1]); kw.setdefault("total_reward", self.total_reward)
        if "Vdig" in kw and "Vmove" in kw:
            vd, vm = kw["Vdig"], kw["Vmove"]
            kw["alpha"] = float(vd - vm) if not (np.isnan(vd) or np.isnan(vm)) else np.nan
        self.log.append(kw)

    def _snapshot(self, action: str, decision: str, Vdig: float, Vmove: float, w: float):
        self.frames_pos.append(tuple(self.pos))
        self.frames_action.append(action); self.frames_decision.append(decision)
        self.frames_Vdig.append(float(Vdig)); self.frames_Vmove.append(float(Vmove)); self.frames_w.append(float(w))
        self.frames_total_reward.append(float(self.total_reward))

    def _stay_and_collect(self) -> float:
        r = self.current_reward_prob(); self.total_reward += r; return r

    def _move_to_best_A(self, w: float) -> Tuple[bool, int]:
        """Returns (moved, alien_level_at_new_cell). No extra log here - caller logs once."""
        nbrs = self.neighbors()
        if not nbrs: return False, 0
        best_val, best_p = -np.inf, None
        for p in nbrs:
            a = self.A_goal(p, w)
            d = abs(p[0]-self.security_pos[0]) + abs(p[1]-self.security_pos[1])
            v = self.cfg.lambda_sec * float(d) * a - self.cfg.move_cost
            if v > best_val: best_val, best_p = v, p
        if best_p is None: return False, 0
        self.pos = best_p; self.visited.add(self.pos)
        al = int(self.env.loc[self.pos, "alien_level"])
        if al > 0 and self.pos != self.security_pos:
            self.stunned_steps = 3
        return True, al

    def step(self):
        """One step: one _log and one _snapshot at end, so CSV and GIF stay in sync."""
        if self.stunned_steps > 0:
            self._log(action="stunned", decision="stunned", Vdig=np.nan, Vmove=np.nan, w=np.nan)
            self._snapshot("stunned", "stunned", np.nan, np.nan, np.nan)
            self.stunned_steps -= 1; self.t += 1; return
        if self.current_reward_prob() > 0.0:
            Vd, Vm = self.Vdig(), self.Vmove(); w = self.w_t(Vd, Vm)
            self._stay_and_collect()
            self._log(action="stay", decision="stay", Vdig=Vd, Vmove=Vm, w=w, r_gained=self.current_reward_prob())
            self._snapshot("stay", "stay", Vd, Vm, w); self.t += 1; return
        Vd, Vm = self.Vdig(), self.Vmove(); w = self.w_t(Vd, Vm)
        force = (self.current_reward_prob() == 0.0 and
                 (any(self.env.loc[p,"reward_prob"]>0 for p in self.neighbors()) or any(p not in self.visited for p in self.env.index)))
        if (Vd > Vm) and not force:
            self._stay_and_collect()
            self._log(action="stay", decision="stay", Vdig=Vd, Vmove=Vm, w=w)
            self._snapshot("stay", "stay", Vd, Vm, w)
        else:
            moved, alien_lvl = self._move_to_best_A(w)
            self._log(action="move" if moved else "no_move", decision="move", Vdig=Vd, Vmove=Vm, w=w, alien_level=alien_lvl if alien_lvl else None)
            self._snapshot("move" if moved else "no_move", "move", Vd, Vm, w)
        self.t += 1

    def run(self) -> pd.DataFrame:
        for _ in range(self.cfg.total_moves): self.step()
        return pd.DataFrame(self.log)

def run_sim_on_env(env: pd.DataFrame, seed: Optional[int] = None) -> Dict[str, object]:
    cfg = ForagerConfig(total_moves=40, lambda_sec=1.0, w_scale=1.0, move_cost=0.0, security_pos=None, seed=seed)
    agent = ForagerAgent(env, cfg)
    log_df = agent.run()
    return {"log": log_df, "total_reward": agent.total_reward, "agent": agent}

def animate_forager(env: pd.DataFrame, agent: ForagerAgent, outpath: str = "forager.gif"):
    env_idx = env.set_index(["Row", "Col"], drop=False)
    max_r, max_c = int(env_idx["Row"].max()), int(env_idx["Col"].max())
    base = np.zeros((max_r+1, max_c+1))
    for (r, c), row in env_idx.iterrows(): base[r, c] = row["reward_prob"]
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(base, cmap="YlOrRd", origin="upper", vmin=0, vmax=1)
    dot, = ax.plot([], [], "bo", markersize=10)
    ax.set_xticks(range(max_c+1)); ax.set_yticks(range(max_r+1)); ax.set_title("Forager leader (blue dot); stunned = no move"); ax.grid(True, linestyle=":", alpha=0.3)
    def init(): dot.set_data([], []); return [im, dot]
    def update(f): r, c = agent.frames_pos[f]; dot.set_data([c], [r]); return [im, dot]
    FuncAnimation(fig, update, init_func=init, frames=len(agent.frames_pos), interval=400, blit=True).save(outpath, writer=PillowWriter(fps=4))
    plt.close(fig)

# ----- Run: leader forager, move + stun -----
single_csv = os.path.join(CSV_DIR, "middle_reward_middle_risk_01.csv")
if not os.path.isfile(single_csv):
    raise FileNotFoundError(f"CSV not found: {single_csv}\nPut middle_reward_middle_risk_01.csv in {os.path.abspath(CSV_DIR)} or upload it to Colab (Files panel) and set CSV_DIR = '.'")
env = load_grid_from_csv(single_csv)
stats = run_sim_on_env(env, seed=0)

print("Total reward:", stats["total_reward"])
print("\nLog (action, decision, row, col, total_reward):")
print(stats["log"][["step", "action", "decision", "row", "col", "total_reward"]].to_string())

stats["log"].to_csv("forager_single_01.csv", index=False)
animate_forager(env, stats["agent"], outpath="forager_01.gif")
print("\nSaved forager_single_01.csv and forager_01.gif")


Total reward: 21.60000000000001

Log (action, decision, row, col, total_reward):
    step action decision  row  col  total_reward
0      0   move     move    5    6           0.0
1      1   move     move    4    6           0.0
2      2   move     move    3    6           0.0
3      3   move     move    2    6           0.0
4      4   move     move    1    6           0.0
5      5   move     move    0    6           0.0
6      6   move     move    0    5           0.0
7      7   move     move    0    4           0.0
8      8   move     move    0    3           0.0
9      9   move     move    0    2           0.0
10    10   move     move    0    1           0.0
11    11   move     move    0    0           0.0
12    12   move     move    1    0           0.0
13    13   stay     stay    1    0           0.8
14    14   stay     stay    1    0           1.6
15    15   stay     stay    1    0           2.4
16    16   stay     stay    1    0           3.2
17    17   stay     stay    1    0   

forager_01.gif	       gridworld   js		Util_ipynb
forager_single_01.csv  index.html  TexturePack


In [ ]:
!git add Role_Specified_Collective_Foraging_Task_Model_Code.ipynb
!git commit -m "update experiment"
!git push

fatal: pathspec 'Role_Specified_Collective_Foraging_Task_Model_Code.ipynb' did not match any files
On branch master

Initial commit

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.DS_Store
	TexturePack/
	Util_ipynb/
	forager_01.gif
	forager_single_01.csv
	gridworld/
	index.html
	js/

nothing added to commit but untracked files present (use "git add" to track)
fatal: The current branch master has no upstream branch.
To push the current branch and set the remote as upstream, use

    git push --set-upstream origin master

